# Network Intrusion Detection — NNConv Semi-Supervised Anomaly Detector
Full pipeline: data loading → subsampling → label-aware edge split → negative sampling → PyG Data → NNConv model → anomaly evaluation

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import NNConv

from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Helper functions — parse GraphML

In [ ]:
def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph."""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)


def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}
        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except Exception:
                    edge_attrs[attr_name] = data_elem.text
        G.add_edge(source, target, **edge_attrs)

    return G


def extract_edge_features(G):
    """Extract edge features as a pandas DataFrame."""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {'source': u, 'target': v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)

## 3. Load graph

In [ ]:
FILE_PATH = "/Users/dannykim/Desktop/vscode/Comp 395/Network-Intrusion-GNN/archive/0.1M-Stratified-Multi.graphml"

G_full = parse_graphml(FILE_PATH)
print(f"Full graph  — nodes: {G_full.number_of_nodes():,}  edges: {G_full.number_of_edges():,}")

# Extract edge features from the full graph for later use
edge_df = extract_edge_features(G_full)
print("\nEdge feature columns:", edge_df.columns.tolist())
print(edge_df.describe())

## 4. Subsample to a manageable size

The full graph has 41k nodes. We BFS-subsample to ~3000 nodes for development.

In [ ]:


def get_dense_subgraph(G_full, target_nodes=2000):
    # 1. Pick a random starting node that actually has connections
    random.seed(SEED)
    start_node = random.choice([n for n, d in G_full.degree() if d > 5])
    
    # 2. Use BFS to find the closest 2000 nodes
    nodes = {start_node}
    queue = [start_node]
    
    while len(nodes) < target_nodes and queue:
        current = queue.pop(0)
        for neighbor in G_full.neighbors(current):
            if neighbor not in nodes:
                nodes.add(neighbor)
                queue.append(neighbor)
                if len(nodes) >= target_nodes:
                    break
                    
    # 3. Create the induced subgraph
    return G_full.subgraph(nodes).copy()

# Update your Section 4 with this:
G_sub = get_dense_subgraph(G_full, target_nodes=3000)
G_int = nx.convert_node_labels_to_integers(G_sub)

print(f"Subgraph — nodes: {G_sub.number_of_nodes():,}  edges: {G_sub.number_of_edges():,}")

print(f"Relabelled  — nodes: {G_int.number_of_nodes():,}  edges: {G_int.number_of_edges():,}")

## 5. Edge train / val / test split

Semi-supervised methodology: model trains ONLY on normal edges.
Attack edges are withheld entirely until final evaluation.

In [ ]:
def preview_graph_edges(G, name, n=5):
    edges = list(G.edges(data=True))[:n]
    print(f"--- {name} (first {n} edges) ---")
    for u, v, edata in edges:
        # MultiGraph: edata is {0: {attrs}, 1: {attrs}, ...}
        inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
        label = inner.get('ActivityLabel', 'N/A')
        print(f"  ({u}, {v}) → ActivityLabel: {label}")


# ── Separate edges by label ───────────────────────────────────────────────────
normal_edges = []
attack_edges = []

for u, v, edata in G_int.edges(data=True):
    inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
    label = inner.get('ActivityLabel', 0)
    if label == 0:
        normal_edges.append((u, v))
    else:
        attack_edges.append((u, v))

print(f"Normal edges: {len(normal_edges):,}")
print(f"Attack edges: {len(attack_edges):,}")

# ── Split normal edges into train / val / test ────────────────────────────────
random.seed(SEED)
random.shuffle(normal_edges)

n             = len(normal_edges)
train_split   = int(0.70 * n)
val_split     = int(0.85 * n)

train_edges       = normal_edges[:train_split]           # 70% normal — training only
val_edges         = normal_edges[train_split:val_split]  # 15% normal — hyperparameter tuning
test_normal_edges = normal_edges[val_split:]             # 15% normal — held-out for test
test_attack_edges = attack_edges                         # ALL attacks — test only

print(f"Train edges  (normal only): {len(train_edges):,}")
print(f"Val edges    (normal only): {len(val_edges):,}")
print(f"Test normal  (normal only): {len(test_normal_edges):,}")
print(f"Test attack  (attack only): {len(test_attack_edges):,}")
print(f"Test anomaly ratio:         {len(test_attack_edges) / (len(test_normal_edges) + len(test_attack_edges)):.2%}")

# ── Build graphs ──────────────────────────────────────────────────────────────
G_train = nx.MultiGraph()
G_train.add_nodes_from(G_int.nodes(data=True))
G_train.add_edges_from((u, v, G_int[u][v]) for u, v in train_edges)

G_val = nx.MultiGraph()
G_val.add_nodes_from(G_int.nodes(data=True))
G_val.add_edges_from((u, v, G_int[u][v]) for u, v in val_edges)

print(f"G_train — nodes: {G_train.number_of_nodes():,}  edges: {G_train.number_of_edges():,}")
preview_graph_edges(G_train, "G_train")
print(f"G_val   — nodes: {G_val.number_of_nodes():,}  edges: {G_val.number_of_edges():,}")
preview_graph_edges(G_val,   "G_val")

## 6. Negative edge sampling

Random non-existent edges used as contrastive signal during training.
These are NOT attack edges — they are simply absent connections.
This prevents model collapse (outputting 1.0 for everything).

In [ ]:
def sample_negative_edges(num_samples, num_nodes, existing_edges, seed=42):
    """Sample random node pairs that have no edge in G_train."""
    random.seed(seed)
    existing = set(existing_edges) | {(v, u) for u, v in existing_edges}
    neg = []
    attempts = 0
    while len(neg) < num_samples:
        u = random.randint(0, num_nodes - 1)
        v = random.randint(0, num_nodes - 1)
        attempts += 1
        if u != v and (u, v) not in existing:
            neg.append((u, v))
        if attempts > num_samples * 20:
            print(f"Warning: only found {len(neg)} negatives after {attempts} attempts")
            break
    return neg


# Generate after node2idx is built (we use integer indices directly here)
# Placeholder — actual call is in Section 7 after node2idx exists
print("Negative sampler defined — will be called in Section 7 after node2idx is built.")

## 7. Build node index map, integer edge lists, and negative samples

In [ ]:
# node2idx: node_id -> row index in feature matrix x
node2idx = {n: i for i, n in enumerate(G_train.nodes())}


def to_idx(edges):
    """Convert (u, v) node-ID pairs to integer index pairs.
    Drops edges whose endpoints are absent from node2idx.
    """
    return [
        (node2idx[u], node2idx[v])
        for u, v in edges
        if u in node2idx and v in node2idx
    ]


train_pos_idx   = to_idx(train_edges)
val_pos_idx     = to_idx(val_edges)
test_normal_idx = to_idx(test_normal_edges)
test_attack_idx = to_idx(test_attack_edges)

print(f"train_pos_idx:   {len(train_pos_idx):,}")
print(f"val_pos_idx:     {len(val_pos_idx):,}")
print(f"test_normal_idx: {len(test_normal_idx):,}")
print(f"test_attack_idx: {len(test_attack_idx):,}")
print(f"Attack edges dropped (nodes not in training graph): "
      f"{len(test_attack_edges) - len(test_attack_idx):,}")

# ── Negative edges (random non-existent pairs, integer indices) ───────────────
num_nodes = len(node2idx)

neg_train_idx = sample_negative_edges(len(train_pos_idx), num_nodes, train_pos_idx, seed=42)
neg_val_idx   = sample_negative_edges(len(val_pos_idx),   num_nodes, train_pos_idx, seed=43)

print(f"neg_train_idx: {len(neg_train_idx):,}")
print(f"neg_val_idx:   {len(neg_val_idx):,}")

## 8. Build PyG Data object with edge features

Node features: mean of connected edge traffic features (7-dim).
Edge features: per-edge traffic columns used by NNConv.

In [ ]:
EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']
NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7

# ── Node features: mean of incident edge traffic features ─────────────────────
node_feat_dict = {n: [] for n in G_train.nodes()}
for u, v, edata in G_train.edges(data=True):
    for key in edata:  # iterate parallel edges (MultiGraph)
        inner = edata[key]
        feats = [float(inner.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
        node_feat_dict[u].append(feats)
        node_feat_dict[v].append(feats)

node_feats = []
for n in G_train.nodes():
    ef = node_feat_dict[n]
    node_feats.append(np.mean(ef, axis=0) if ef else np.zeros(NUM_EDGE_FEATS))

node_feats = np.array(node_feats)
node_feats = (node_feats - node_feats.mean(axis=0)) / (node_feats.std(axis=0) + 1e-9)
x = torch.tensor(node_feats, dtype=torch.float)  # [N, 7]

# ── Edge index: training graph structure for message passing ──────────────────
train_edge_list = list(G_train.edges())

edge_index = torch.tensor(
    [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
    dtype=torch.long
).t().contiguous()  # [2, E]

# ── Edge attributes: per-edge traffic features ────────────────────────────────
edge_attr = []
for u, v in train_edge_list:
    raw_data = G_train[u][v][0]  # first parallel edge's attributes
    feats = [float(raw_data.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    edge_attr.append(feats)

edge_attr = torch.tensor(edge_attr, dtype=torch.float)  # [E, 7]

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

print("Data object:")
print(f"  x:          {data.x.shape}   (nodes x node_features)")
print(f"  edge_index: {data.edge_index.shape}  (2 x training_edges)")
print(f"  edge_attr:  {data.edge_attr.shape}  (training_edges x edge_features)")

## 9. NNConv model

Encoder: two NNConv layers produce node embeddings.
Decoder: dot product similarity between endpoint embeddings scores each edge.
Higher score = model thinks the edge looks like normal traffic.

In [ ]:
HIDDEN = 64


class GNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden):
        super().__init__()

        # Conv 1: node_in -> hidden
        self.conv1 = NNConv(
            in_channels=node_in,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, node_in * hidden),
                nn.ReLU()
            )
        )

        # Conv 2: hidden -> hidden
        self.conv2 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )

    def encode(self, x, edge_index, edge_attr):
        """Produce node embeddings via message passing."""
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = self.conv2(x, edge_index, edge_attr)
        return x

    def decode(self, z, edges):
        """Score candidate edges — higher = more normal."""
        u = torch.tensor([e[0] for e in edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in edges], dtype=torch.long)
        return (z[u] * z[v]).sum(dim=1)

    def forward(self, x, edge_index, edge_attr, edges):
        z = self.encode(x, edge_index, edge_attr)
        return self.decode(z, edges)


model = GNN(
    node_in=x.shape[1],       # 7
    edge_in=NUM_EDGE_FEATS,   # 7
    hidden=HIDDEN
)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 10. Training

Contrastive loss:
- Push normal edge scores toward 1 (pos_loss)
- Push random non-edge scores toward 0 (neg_loss)

The model never sees attack edges during training.

In [ ]:
def evaluate(model, data, val_normal_idx, val_neg_idx):
    """Validation: AUROC separating normal edges from random non-edges."""
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)

        all_edges = val_normal_idx + val_neg_idx
        u = torch.tensor([e[0] for e in all_edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in all_edges], dtype=torch.long)
        scores = torch.sigmoid((z[u] * z[v]).sum(dim=1)).numpy()

    y_true = [1] * len(val_normal_idx) + [0] * len(val_neg_idx)
    auc = roc_auc_score(y_true, scores)
    return auc


def evaluate_test(model, data, test_normal_idx, test_attack_idx):
    """Final test: AUROC and AUPRC separating normal from attack edges."""
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)

        all_edges = test_normal_idx + test_attack_idx
        u = torch.tensor([e[0] for e in all_edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in all_edges], dtype=torch.long)
        scores = torch.sigmoid((z[u] * z[v]).sum(dim=1)).numpy()

   
    anomaly_scores = scores
    y_true = [0] * len(test_normal_idx) + [1] * len(test_attack_idx)

    auc = roc_auc_score(y_true, anomaly_scores)
    ap  = average_precision_score(y_true, anomaly_scores)
    return auc, ap


# ── Training loop ─────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 500

u_pos = torch.tensor([e[0] for e in train_pos_idx], dtype=torch.long)
v_pos = torch.tensor([e[1] for e in train_pos_idx], dtype=torch.long)
u_neg = torch.tensor([e[0] for e in neg_train_idx], dtype=torch.long)
v_neg = torch.tensor([e[1] for e in neg_train_idx], dtype=torch.long)

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    z = model.encode(data.x, data.edge_index, data.edge_attr)

    # Positive: normal edges should score HIGH
    pos_scores = (z[u_pos] * z[v_pos]).sum(dim=1)
    pos_loss   = (1 - torch.sigmoid(pos_scores)).mean()

    # Negative: random non-edges should score LOW
    neg_scores = (z[u_neg] * z[v_neg]).sum(dim=1)
    neg_loss   = torch.sigmoid(neg_scores).mean()

    loss = pos_loss + neg_loss
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        val_auc = evaluate(model, data, val_pos_idx, neg_val_idx)
        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Val AUROC: {val_auc:.4f}")

## 11. Final test evaluation

Score test edges using anomaly score = 1 - similarity.
Low similarity → model thinks the edge is abnormal → flagged as attack.

In [ ]:
test_auc, test_ap = evaluate_test(model, data, test_normal_idx, test_attack_idx)
print(f"Test AUROC: {test_auc:.4f}")
print(f"Test AUPRC: {test_ap:.4f}")

## 12. Sanity checks

In [ ]:
import copy

model.eval()
with torch.no_grad():
    z = model.encode(data.x, data.edge_index, data.edge_attr)
    all_edges = test_normal_idx + test_attack_idx
    u = torch.tensor([e[0] for e in all_edges], dtype=torch.long)
    v = torch.tensor([e[1] for e in all_edges], dtype=torch.long)
    scores = torch.sigmoid((z[u] * z[v]).sum(dim=1)).numpy()

anomaly_scores = 1 - scores
y_true     = [0] * len(test_normal_idx) + [1] * len(test_attack_idx)
y_shuffled = copy.copy(y_true)
random.shuffle(y_shuffled)

print("Real AUROC:    ", roc_auc_score(y_true, anomaly_scores))
print("Shuffled AUROC:", roc_auc_score(y_shuffled, anomaly_scores))

# Degree baseline: high-degree nodes = more normal traffic
deg = dict(G_train.degree())
deg_scores = [-(deg.get(u, 0) + deg.get(v, 0)) for u, v in all_edges]
print("Degree baseline AUROC:", roc_auc_score(y_true, deg_scores))